# Notebook 10 kernel encoder definitions

Purpose: define spatial and learned-coordinate dense positive kernel encoders that replace the context and EMA target ViT encoders in the existing JEPA objective. Inputs are image tensors, visible original patch indices and validity masks; outputs retain `[batch, visible patches, encoder width]`. The default is four residual/MLP blocks of width 192, three heads and eight coordinates per head.

Coordinates are learned embeddings, **not a proved data manifold**. The first two coordinates remain anchored patch positions. The remaining coordinates depend on the supplied visible patch features and positions. Query and source maps are separate; they start equal with nonzero weights so geometry receives useful gradients immediately. Smooth Gaussian-mixture sums are evaluated densely and exactly up to floating-point arithmetic. This is a representation baseline, **not an FMM implementation or acceleration result**. Mixture weights are softmax-normalized and bandwidths are constrained to `[0.03, 2]`.

Run All imports definitions and reports the API only; no training or dataset loading occurs here. Expected runtime: less than one minute. The consuming study notebook owns outputs and checkpoint/resume policy under `checkpoints/<run_id>/`. These definitions carry no run state: restart this notebook from its first cell, or import only the explicitly tagged `ahr-export` cells through the project notebook bootstrap. Validation is in `tests/10_Kernel_Encoder_Validation.ipynb`.


In [ ]:
from pathlib import Path
import json, os
ROOT=Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_bootstrap={}
for _cell in json.loads((ROOT/"fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"]=="code" and "ahr-export" in _cell.get("metadata",{}).get("tags",[]):
        exec("".join(_cell["source"]), _bootstrap)
_bootstrap["install_notebook_package"](ROOT)


In [ ]:
"""Dense smooth positive kernel encoders with optional learned token coordinates."""
import copy
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from fmm_jepa.models import JEPA, Predictor, patch_coordinates, position_features, validate_positions

BACKEND = "dense-learned-kernel-v1"
VARIANTS = ("vit", "spatial_kernel", "learned_kernel")


class NormalizedRBFMixer(nn.Module):
    """Exact dense normalized Gaussian-mixture interaction; no FMM approximation.

    The first two coordinates are anchored image positions. Learned variants add
    bounded feature/position coordinates, independently for queries and sources.
    Their maps start equal but have separate trainable storage and nonzero weights.
    """
    def __init__(self, dim, heads=3, variant="learned_kernel", coordinate_dim=8,
                 bandwidths=(.15, .35, .75), content_scale=.5):
        super().__init__()
        if variant not in ("spatial_kernel", "learned_kernel"):
            raise ValueError("Expected spatial_kernel or learned_kernel")
        if min(dim, heads) < 1 or dim % heads:
            raise ValueError("Positive dimension divisible by positive heads required")
        if coordinate_dim < 3:
            raise ValueError("coordinate_dim must reserve two spatial and >=1 content dimensions")
        bandwidths = tuple(float(value) for value in bandwidths)
        if not bandwidths or any(not math.isfinite(value) or not .03 <= value <= 2. for value in bandwidths):
            raise ValueError("Initial bandwidths must be finite and in [.03, 2]")
        if not math.isfinite(content_scale) or content_scale <= 0:
            raise ValueError("Finite positive content_scale required")
        self.dim, self.heads, self.variant = int(dim), int(heads), variant
        self.coordinate_dim, self.content_scale = int(coordinate_dim), float(content_scale)
        self.value = nn.Linear(dim, dim)
        self.projection = nn.Linear(dim, dim)
        self.log_bandwidths = nn.Parameter(torch.tensor(bandwidths).log().expand(heads, -1).clone())
        self.mixture_logits = nn.Parameter(torch.zeros(heads, len(bandwidths)))
        if variant == "learned_kernel":
            extra = heads * (coordinate_dim - 2)
            self.query_content, self.key_content = nn.Linear(dim, extra), nn.Linear(dim, extra)
            self.query_position, self.key_position = nn.Linear(2, extra, bias=False), nn.Linear(2, extra, bias=False)
            nn.init.normal_(self.query_content.weight, std=.02)
            nn.init.zeros_(self.query_content.bias)
            nn.init.normal_(self.query_position.weight, std=.02)
            with torch.no_grad():
                self.key_content.load_state_dict(self.query_content.state_dict())
                self.key_position.load_state_dict(self.query_position.state_dict())
        self.backend = "exact dense normalized positive RBF mixture; quadratic token pairs"

    def _coordinates(self, x, indices, valid, grid):
        # Explicit math precision avoids half-precision squared-distance cancellation.
        math_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
        safe = torch.where(valid[..., None], x, torch.zeros_like(x)).to(math_dtype)
        xy = patch_coordinates(indices.clamp(0, grid[0] * grid[1] - 1), grid, dtype=math_dtype)
        spatial = xy[:, None].expand(-1, self.heads, -1, -1)
        if self.variant == "spatial_kernel":
            extra = spatial.new_zeros((*spatial.shape[:-1], self.coordinate_dim - 2))
            return torch.cat((spatial, extra), -1), torch.cat((spatial, extra), -1)
        def transform(content, position):
            extra = F.linear(safe, content.weight.to(math_dtype), content.bias.to(math_dtype))
            extra = extra + F.linear(xy, position.weight.to(math_dtype))
            extra = self.content_scale * extra.tanh()
            extra = extra.reshape(x.shape[0], x.shape[1], self.heads, self.coordinate_dim - 2).transpose(1, 2)
            return torch.cat((spatial, extra), -1)
        return transform(self.query_content, self.query_position), transform(self.key_content, self.key_position)

    def _log_kernel(self, query_coordinates, source_coordinates):
        dtype = query_coordinates.dtype
        squared_distance = (query_coordinates.square().sum(-1)[..., :, None]
                            + source_coordinates.square().sum(-1)[..., None, :]
                            - 2 * (query_coordinates @ source_coordinates.transpose(-1, -2))).clamp_min(0)
        # Bandwidth bounds are part of the declared model and protect degenerate scales.
        bandwidths = self.log_bandwidths.to(dtype).clamp(math.log(.03), math.log(2.)).exp()
        log_mixture = self.mixture_logits.to(dtype).log_softmax(-1)
        terms = [log_mixture[None, :, scale, None, None]
                 - squared_distance / (2 * bandwidths[None, :, scale, None, None].square())
                 for scale in range(bandwidths.shape[-1])]
        return torch.logsumexp(torch.stack(terms, 0), dim=0)

    def kernel_state(self, x, indices, valid, grid):
        """Coordinates and exactly row-normalized kernel, useful for bounded audits."""
        with torch.autocast(device_type=x.device.type, enabled=False):
            query, source = self._coordinates(x, indices, valid, grid)
            log_kernel = self._log_kernel(query, source)
            logits = log_kernel.masked_fill(~valid[:, None, None, :], float("-inf"))
            weights = logits.softmax(-1)
            weights = torch.where(valid[:, None, :, None], weights, torch.zeros_like(weights))
        return {"query_coordinates": query, "source_coordinates": source,
                "log_kernel": log_kernel, "weights": weights}

    def forward(self, x, indices=None, valid=None, grid=None):
        if indices is None or valid is None or grid is None:
            raise ValueError("Original patch indices, validity and grid are required")
        if x.ndim != 3 or x.shape[-1] != self.dim or x.shape[:2] != indices.shape or valid.shape != indices.shape:
            raise ValueError("Expected matching x[B,V,D], indices[B,V] and valid[B,V]")
        if not bool(valid.any(-1).all()):
            raise ValueError("Each sample must have at least one valid source")
        x = torch.where(valid[..., None], x, torch.zeros_like(x))
        values = self.value(x).reshape(x.shape[0], x.shape[1], self.heads, self.dim // self.heads).transpose(1, 2)
        state = self.kernel_state(x, indices, valid, grid)
        with torch.autocast(device_type=x.device.type, enabled=False):
            mixed = state["weights"] @ values.to(state["weights"].dtype)
        mixed = mixed.transpose(1, 2).reshape_as(x).to(values.dtype)
        return torch.where(valid[..., None], self.projection(mixed), torch.zeros_like(x))


class KernelEncoderBlock(nn.Module):
    def __init__(self, variant, dim, heads=3, **options):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
        self.mixer = NormalizedRBFMixer(dim, heads=heads, variant=variant, **options)

    def forward(self, x, indices, valid, grid):
        x = x + self.mixer(self.norm1(x), indices, valid, grid)
        x = x + self.mlp(self.norm2(x))
        return torch.where(valid[..., None], x, torch.zeros_like(x))


class KernelVisionEncoder(nn.Module):
    """Mask-safe ViT-shaped stem/residual/MLP scaffold, returning [B,V,D]."""
    def __init__(self, kind="learned_kernel", patch_size=8, dim=192, depth=4, heads=3,
                 coordinate_dim=8, bandwidths=(.15, .35, .75), content_scale=.5, checkpoint=False):
        super().__init__()
        if kind not in ("spatial_kernel", "learned_kernel"):
            raise ValueError("Expected spatial_kernel or learned_kernel")
        if min(patch_size, dim, depth) < 1:
            raise ValueError("Positive patch_size, dimension and depth required")
        self.patch_size, self.dim, self.kind = int(patch_size), int(dim), kind
        self.checkpoint_blocks = bool(checkpoint)
        self.patch_embed = nn.Linear(3 * patch_size * patch_size, dim)
        self.blocks = nn.ModuleList([KernelEncoderBlock(kind, dim, heads=heads, coordinate_dim=coordinate_dim,
            bandwidths=bandwidths, content_scale=content_scale) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def embed(self, images, indices=None, valid=None):
        if images.ndim != 4 or images.shape[1] != 3:
            raise ValueError("images must have shape [B,3,H,W]")
        batch, _, height, width = images.shape
        if height % self.patch_size or width % self.patch_size:
            raise ValueError("Image dimensions must be divisible by patch_size")
        grid = (height // self.patch_size, width // self.patch_size)
        count = grid[0] * grid[1]
        if indices is None:
            if valid is not None:
                raise ValueError("valid cannot be supplied without indices")
            indices = torch.arange(count, device=images.device).expand(batch, -1)
        if valid is None:
            valid = torch.ones_like(indices, dtype=torch.bool)
        if images.device != indices.device or images.device != valid.device:
            raise ValueError("Images, indices and validity must share a device")
        validate_positions(indices, valid, batch, count)
        # Only gather visible raw pixels before ANY learned transform; hidden NaNs are safe.
        raw = F.unfold(images, kernel_size=self.patch_size, stride=self.patch_size).transpose(1, 2)
        gather = indices.clamp(0, count - 1)[..., None].expand(-1, -1, raw.shape[-1])
        visible_pixels = torch.where(valid[..., None], raw.gather(1, gather), torch.zeros_like(raw.gather(1, gather)))
        x = self.patch_embed(visible_pixels)
        x = torch.where(valid[..., None], x + position_features(indices.clamp(0, count - 1), grid, self.dim, x.dtype), torch.zeros_like(x))
        return x, indices, valid, grid

    def forward(self, images, indices=None, valid=None):
        x, indices, valid, grid = self.embed(images, indices, valid)
        for block in self.blocks:
            if self.checkpoint_blocks and self.training and torch.is_grad_enabled():
                x = activation_checkpoint(block, x, indices, valid, grid, use_reentrant=False)
            else:
                x = block(x, indices, valid, grid)
        return torch.where(valid[..., None], self.norm(x), torch.zeros_like(x))


class KernelJEPA(JEPA):
    """Replace both context/EMA encoders; inherit the audited predictor and objective."""
    def __init__(self, config, variant=None):
        nn.Module.__init__(self)
        self.config = copy.deepcopy(config)
        options = config.get("kernel10", {})
        variant = variant or options.get("variant", "learned_kernel")
        self.context = KernelVisionEncoder(kind=variant, patch_size=config.get("patch_size", 8),
            dim=config.get("dim", 192), depth=options.get("depth", config.get("depth", 4)),
            heads=options.get("heads", config.get("heads", 3)), coordinate_dim=options.get("coordinate_dim", 8),
            bandwidths=options.get("bandwidths", (.15, .35, .75)), content_scale=options.get("content_scale", .5),
            checkpoint=options.get("checkpoint", False))
        self.target = copy.deepcopy(self.context).requires_grad_(False).eval()
        self.predictor = Predictor(self.context.dim, dim=config.get("predictor_dim", 128),
            depth=config.get("predictor_depth", 2), heads=config.get("predictor_heads", 4))


def build_model(config, kind="semantic", variant=None):
    variant = variant or config.get("kernel10", {}).get("variant", "learned_kernel")
    if kind == "vit" or variant == "vit":
        cfg = copy.deepcopy(config)
        cfg["kind"] = "vit"
        return JEPA(cfg)
    if kind != "semantic":
        raise ValueError("kind must be vit or semantic")
    return KernelJEPA(config, variant=variant)


def parameter_counts(model):
    count = lambda module: sum(parameter.numel() for parameter in module.parameters())
    return {"variant": model.context.kind, "encoder_total": count(model.context),
            "encoder_inactive_scaffold": 0, "encoder_active": count(model.context),
            "predictor": count(model.predictor),
            "trainable_total": sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad),
            "trainable_active": sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad),
            "with_ema_teacher_total": count(model)}


## Bounded diagnostics

The diagnostic helper measures coordinate spread, query/source displacement, normalized kernel entropy and effective neighbors. It also reports relative SVD approximation errors for predeclared opposite spatial corner blocks when their actual learned-coordinate bounding balls are well separated. These local measurements do not establish global low rank, a manifold, an FMM approximation guarantee or a speed benefit. Geometry setup and all dense computation remain inside the actual context/EMA encoder path.


In [ ]:
@torch.no_grad()
def kernel_geometry_diagnostics(model_or_encoder, images, indices=None, valid=None, max_samples=2,
                                far_block_ranks=(8, 16, 32), separation_factor=1.25):
    """Bounded descriptive geometry/compressibility diagnostics, NOT speed evidence.

    Far blocks use two predeclared opposite spatial corner sets. A block qualifies
    only when its learned coordinate bounding balls meet the stated separation
    factor. Relative SVD tail errors describe this block, not a full FMM scheme.
    """
    encoder = getattr(model_or_encoder, "context", model_or_encoder)
    if not isinstance(encoder, KernelVisionEncoder):
        return {"status": "NOT_APPLICABLE", "reason": "Not a kernel encoder"}
    limit = min(images.shape[0], int(max_samples))
    if limit < 1:
        raise ValueError("max_samples must be positive")
    images = images[:limit]
    if indices is not None: indices = indices[:limit]
    if valid is not None: valid = valid[:limit]
    prior_mode = encoder.training
    encoder.eval()
    try:
        x, indices, valid, grid = encoder.embed(images, indices, valid)
        layers = []
        for block_index, block in enumerate(encoder.blocks):
            state = block.mixer.kernel_state(block.norm1(x), indices, valid, grid)
            q, k, weights = state["query_coordinates"], state["source_coordinates"], state["weights"]
            row_mask = valid[:, None, :].expand(-1, block.mixer.heads, -1)
            entropy = -(weights * weights.clamp_min(torch.finfo(weights.dtype).tiny).log()).sum(-1)[row_mask]
            participation = 1 / weights.square().sum(-1)[row_mask]
            coordinate_rows = q.transpose(1, 2)[valid]
            source_rows = k.transpose(1, 2)[valid]
            row = {"block": block_index, "variant": encoder.kind,
                   "coordinate_dimension": block.mixer.coordinate_dim,
                   "coordinate_spread_mean_std": float(coordinate_rows.std(dim=0, unbiased=False).mean()),
                   "content_coordinate_rms": float(coordinate_rows[..., 2:].square().mean().sqrt()),
                   "query_source_displacement_mean": float((coordinate_rows - source_rows).norm(dim=-1).mean()),
                   "kernel_entropy_mean": float(entropy.mean()),
                   "effective_neighbors_exp_entropy_mean": float(entropy.exp().mean()),
                   "effective_neighbors_participation_mean": float(participation.mean()),
                   "bandwidths": block.mixer.log_bandwidths.clamp(math.log(.03), math.log(2.)).exp().cpu().tolist(),
                   "mixture_weights": block.mixer.mixture_logits.softmax(-1).cpu().tolist(),
                   "far_blocks": []}
            for batch_index in range(limit):
                xy = patch_coordinates(indices[batch_index].clamp(0, grid[0] * grid[1] - 1), grid)
                receiver = ((xy[:, 0] < .35) & (xy[:, 1] < .35) & valid[batch_index]).nonzero().flatten()
                source = ((xy[:, 0] > .65) & (xy[:, 1] > .65) & valid[batch_index]).nonzero().flatten()
                for head in range(block.mixer.heads):
                    record = {"sample": batch_index, "head": head,
                              "receiver_count": int(receiver.numel()), "source_count": int(source.numel()),
                              "separation_factor_required": float(separation_factor)}
                    if receiver.numel() < 2 or source.numel() < 2:
                        record.update(status="INSUFFICIENT_POINTS")
                    else:
                        qa, kb = q[batch_index, head, receiver], k[batch_index, head, source]
                        ca, cb = qa.mean(0), kb.mean(0)
                        radii = (qa - ca).norm(dim=-1).max() + (kb - cb).norm(dim=-1).max()
                        separation = (ca - cb).norm()
                        record.update(center_distance=float(separation), sum_bounding_radii=float(radii),
                                      separation_ratio=float(separation / radii.clamp_min(1e-12)))
                        if separation <= separation_factor * radii:
                            record.update(status="NOT_WELL_SEPARATED")
                        else:
                            log_block = state["log_kernel"][batch_index, head][receiver][:, source]
                            # A common scale cancels from relative errors and avoids underflow.
                            matrix = (log_block - log_block.max()).exp().double()
                            singular = torch.linalg.svdvals(matrix)
                            total = singular.square().sum().clamp_min(torch.finfo(singular.dtype).tiny)
                            record.update(status="MEASURED", relative_frobenius_tail_error={
                                str(int(rank)): float((singular[int(rank):].square().sum() / total).sqrt())
                                for rank in far_block_ranks if int(rank) > 0})
                    row["far_blocks"].append(record)
            layers.append(row)
            x = block(x, indices, valid, grid)
        return {"status": "MEASURED", "variant": encoder.kind, "grid": list(grid),
                "samples": limit, "layers": layers, "dense_only": True,
                "interpretation": "Descriptive coordinate/kernel diagnostics; no manifold or FMM speedup established."}
    finally:
        encoder.train(prior_mode)


In [ ]:
print({"backend": BACKEND, "variants": VARIANTS, "builder": "build_model(config, kind=semantic, variant=None)", "config_options": "kernel10", "training_started": False})